# SWaT / WADI：正式三组 C3 × 三随机种子（99号）

这是 99 号正式主结果，不是 quick validation。每个数据集比较 baseline、restricted C3、prototype-query C3；随机种子为 3407、3408、3409，最大 30 epochs、patience=3。代码从 GitHub 主分支获取，GPU 使用 NVIDIA T4。

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, torch
assert torch.cuda.is_available(), '请为 notebook 启用 GPU'
work_root = Path('/kaggle/working/EnhancedMTADGAT')
repo_url = 'https://github.com/wonkawonka/EnhancedMTADGAT.git'
repo_ref = 'main'
if work_root.exists():
    shutil.rmtree(work_root)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', repo_ref, repo_url, str(work_root)], check=True)
os.chdir(work_root)
assert (work_root / 'configs/internal/99_swat_wadi_c3_three_group_three_seed.json').is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('Source:', repo_url, repo_ref)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle-main.txt'], check=True)

In [ ]:
input_root = Path('/kaggle/input')
swat_files = list(input_root.rglob('SWaT_Dataset_Normal_v1.xlsx'))
wadi_files = list(input_root.rglob('WADI_14days_new.csv'))
assert len(swat_files) == 1, swat_files
assert len(wadi_files) == 1, wadi_files
swat_root = swat_files[0].parents[2]
wadi_root = wadi_files[0].parents[2]
os.environ['MTAD_GAT_SWAT_ROOT'] = str(swat_root)
os.environ['MTAD_GAT_WADI_ROOT'] = str(wadi_root)
os.environ['MTAD_GAT_DATASETS_ROOT'] = str(work_root / 'datasets')
os.environ['MTAD_GAT_RUNS_ROOT'] = str(work_root / 'runs')
print('SWaT root:', swat_root)
print('WADI root:', wadi_root)

In [ ]:
subprocess.run([sys.executable, 'run.py', 'preprocess', '--dataset', 'SWAT'], check=True)
subprocess.run([sys.executable, 'run.py', 'preprocess', '--dataset', 'WADI'], check=True)
# 训练阶段只读取 /kaggle/working 中刚生成的 processed 产物。
os.environ['MTAD_GAT_SWAT_ROOT'] = str(work_root / 'datasets/SWAT')
os.environ['MTAD_GAT_WADI_ROOT'] = str(work_root / 'datasets/WADI')
assert (Path(os.environ['MTAD_GAT_SWAT_ROOT']) / 'processed/SWAT_train.pkl').is_file()
assert (Path(os.environ['MTAD_GAT_WADI_ROOT']) / 'processed/WADI_train.npy').is_file()

In [ ]:
plan = 'configs/internal/99_swat_wadi_c3_three_group_three_seed.json'
subprocess.run([sys.executable, 'run.py', 'internal', '--plan', plan, '--dry-run'], check=True)
subprocess.run([sys.executable, '-m', 'src.runners.compare_experiments', '--plan', plan, '--batch-tag', 'kaggle_t4_3seed'], check=True)

In [ ]:
import shutil
run_root = Path('runs/internal/99_swat_wadi_c3_three_group_three_seed__kaggle_t4_3seed')
assert (run_root / 'run_registry.json').is_file(), run_root
archive = shutil.make_archive('/kaggle/working/99_swat_wadi_c3_three_group_three_seed__kaggle_t4_3seed', 'zip', run_root)
print('Results:', archive)